In [1]:
import sys

!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U torch --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install optuna

Looking in indexes: https://download.pytorch.org/whl/cu128


In [2]:
import pandas as pd
import optuna
import torch
import Util
import numpy as np

from Util import DatasetGenerator, Visualization, Table
from dataclasses import dataclass
from torch.nn.functional import mse_loss
from torch.nn import Sigmoid
from PIL import Image

from torch.utils.data import random_split, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

C:\Users\Luco1421\Desktop\U\ia\TPs\TP3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = Util.DEVICE
TYPE = Util.TYPE

# Global visualization utility instance
visualization = Visualization()

In [4]:
def xavier_glorot_init(fan_in, fan_out):
    return (6 / (fan_in + fan_out)) ** 0.5

class MultilayerPerceptron:
    def __init__(self, neurons_per_layer : list [int], alpha : float, gamma : float, max_weights : float):
        self.d, self.m, self.k = neurons_per_layer

        self.alpha = alpha
        self.gamma = gamma
        self.max_weights = max_weights

        self.xavier_glorot_o = xavier_glorot_init(self.d, self.m)
        self.xavier_glorot_s = xavier_glorot_init(self.m, self.k)

        self.Wo = torch.empty(self.d, self.m, dtype=TYPE, device=DEVICE).uniform_(-self.xavier_glorot_o, self.xavier_glorot_o)
        self.Ws = torch.empty(self.m, self.k, dtype=TYPE, device=DEVICE).uniform_(-self.xavier_glorot_s, self.xavier_glorot_s)

        self.Bo = torch.zeros(1, self.m, dtype=TYPE, device=DEVICE)
        self.Bs = torch.zeros(1, self.k, dtype=TYPE, device=DEVICE)

        self.delta_Wo = torch.zeros(1, self.m, dtype=TYPE, device=DEVICE)
        self.delta_Ws = torch.zeros(1, self.k, dtype=TYPE, device=DEVICE)

        self.Po = torch.zeros(1, self.m, dtype=TYPE, device=DEVICE)
        self.Yo = torch.zeros(1, self.m, dtype=TYPE, device=DEVICE)
        self.Ps = torch.zeros(1, self.k, dtype=TYPE, device=DEVICE)
        self.Ys = torch.zeros(1, self.k, dtype=TYPE, device=DEVICE)

        self.activation = Sigmoid()

        self.V_Wo = torch.zeros(self.d, self.m, dtype=TYPE, device=DEVICE)
        self.V_Ws = torch.zeros(self.m, self.k, dtype=TYPE, device=DEVICE)
        self.V_Bo = torch.zeros(self.m, dtype=TYPE, device=DEVICE)
        self.V_Bs = torch.zeros(self.k, dtype=TYPE, device=DEVICE)

    def forward(self, X : torch.Tensor):
        # NxM + 1XM
        self.Po = (X @ self.Wo) + self.Bo
        self.Yo = self.activation(self.Po)
        # MxK + 1xK
        self.Ps = (self.Yo @ self.Ws) + self.Bs
        self.Ys = self.activation(self.Ps)

    def backpropagate_deltas(self, T: torch.Tensor):
        # Output layer       NxK                 NxK   =   NxK
        self.delta_Ws = (self.Ys - T) * (self.Ys * (1 - self.Ys))

        # Hidden layer        NxK               KxM                 NxM       =     NxM
        self.delta_Wo = (self.delta_Ws  @ self.Ws.T) * self.Yo * (1 - self.Yo)

    def update_weights(self, X: torch.Tensor):
        # Hidden Layer DxM                                                 DxN       NxM = DxM
        n = X.shape[0]
        self.V_Wo = self.V_Wo * self.gamma + self.alpha * (1 - self.gamma) * (X.T @ self.delta_Wo) / n
        self.Wo = (self.Wo - self.V_Wo).clamp(-self.max_weights, self.max_weights)

        self.V_Bo = self.V_Bo * self.gamma + self.alpha * (1 - self.gamma) * self.delta_Wo.sum(dim=0) / n
        self.Bo = (self.Bo - self.V_Bo).clamp(-self.max_weights, self.max_weights)

        # Output Layer  MxK                                                 MxN                 NxK   =  MxK
        self.V_Ws = self.V_Ws * self.gamma + self.alpha * (1 - self.gamma) * (self.Yo.T @ self.delta_Ws) / n
        self.Ws = (self.Ws - self.V_Ws).clamp(-self.max_weights, self.max_weights)

        self.V_Bs = self.V_Bs * self.gamma + self.alpha * (1 - self.gamma) * self.delta_Ws.sum(dim=0) / n
        self.Bs = (self.Bs - self.V_Bs).clamp(-self.max_weights, self.max_weights)

    def get_error(self, T: torch.Tensor):
        return mse_loss(self.Ys, T).item()

    def get_accuracy(self, T: torch.Tensor):
        return (torch.argmax(self.Ys, dim=1) == torch.argmax(T, dim=1)).float().mean().item()

    def train_mlp(self, nun_epochs: int, X : torch.Tensor, T : torch.Tensor, alpha : float | None = None, gamma : float | None = None):
        if alpha is not None:
            self.alpha = alpha
        if gamma is not None:
            self.gamma = gamma

        history_error = []
        history_accuracy = []
        eps = 1e-5

        for _ in range(nun_epochs):
            self.forward(X)

            history_error.append(self.get_error(T))
            history_accuracy.append(self.get_accuracy(T))

            self.backpropagate_deltas(T)
            self.update_weights(X)

            if len(history_error) > 1 and abs(history_error[-1] - history_error[-2]) < eps:
                break
        self.forward(X)
        history_error.append(self.get_error(T))
        history_accuracy.append(self.get_accuracy(T))

        return history_error, history_accuracy

## Test Forward

In [5]:
@dataclass
class DataXOR:
    X = torch.tensor([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ], dtype=TYPE, device=DEVICE)
    M = 2
    T = torch.tensor([
        [1,0],
        [0,1],
        [0,1],
        [1,0]
    ], dtype=TYPE, device=DEVICE)
    class0 = torch.tensor([
        [0,0],
        [1,1]
    ])
    class1 = torch.tensor([
        [1,0],
        [0,1]
    ])


In [6]:
EPSILON = 0.00001

def objective_xor(trial, epsilon : float = EPSILON):
    alpha = trial.suggest_float("alpha", 0.01, 100, log=True)
    gamma = trial.suggest_float("gamma", 0, 1, log=False)
    mp = MultilayerPerceptron([DataXOR.X.shape[1], DataXOR.M, DataXOR.T.shape[1]], alpha, gamma, 1000)
    history_error, history_accuracy = mp.train_mlp(3000, DataXOR.X, DataXOR.T)

    if history_error[-1] < epsilon:
        accuracy = torch.tensor(history_accuracy)
        error = torch.tensor(history_error)

        visualization.make_decision_boundary_plot(mp.Wo, mp.Ws, mp.Bo, mp.Bs ,-0.5, 1.5,-0.5, 1.5, DataXOR.class0, DataXOR.class1)
        visualization.make_error_plot(history_error, f"Alpha : {alpha} Gamma : {gamma} ")

        trial.set_user_attr("accuracy", torch.where(accuracy > 0.99)[0][0].item())
        trial.set_user_attr("error", torch.where(error < epsilon)[0][0].item())

    return history_error[-1]


def study(objective):
    optuna.logging.set_verbosity(optuna.logging.ERROR)

    stud = optuna.create_study(direction="minimize")
    stud.optimize(objective, n_trials=50)

    optuna.visualization.plot_optimization_history(stud).show()
    optuna.visualization.plot_rank(stud).show()
    optuna.visualization.plot_param_importances(stud).show()
    return stud.trials

def show_best_trial(trials, epsilon : float = EPSILON):
    table = Table(pd.DataFrame(), ["Intento","Alpha", "Gamma", "Value", "Precisión","Error"], "Optimizacion")

    for t in trials:
        if t.value < epsilon:

            ind = table.obtain_row_count()

            table.add(ind, "Intento", t.number)
            table.add(ind, "Alpha", t.params["alpha"])
            table.add(ind, "Gamma", t.params["gamma"])
            table.add(ind, "Value", t.value)

            table.add(ind, "Precisión", t.user_attrs["accuracy"])
            table.add(ind, "Error", t.user_attrs["error"])

    table.show()
    table.latex()

# show_best_trial(study(objective_xor))

In [7]:
class Analyze:
    def __init__(self, seed, func):
        self.data, self.labels = func(seed)
        print(self.data.shape[0])
        visualization.make_scatter(self.data[:,0], self.data[:, 1], self.labels)

        self.config = [[20, 0.9], [20, 0], [2, 0.9], [2, 0]]

        self.class0 = self.data[self.labels == 0]
        self.class1 = self.data[self.labels == 1]

    def run(self, min_x : float, max_x : float, min_y : float, max_y : float):

        test_error = [[] for _ in range(len(self.config))]

        for i in range(4):

            train_data, test_data = DatasetGenerator.split_data(0.2, i + 1, self.data, self.labels)

            for j in range(len(self.config)):
                M, gamma = self.config[j]
                mlp = MultilayerPerceptron([2, M, 1], 0.1, gamma, 1e5) # con a = 200 y mx = 100 es god
                _ , _ = mlp.train_mlp(1000, train_data[0], train_data[1].view(-1,1)) # con 5000 it es mucho mejor

                mlp.forward(test_data[0])
                test_error[j].append(mlp.get_error(test_data[1].view(-1,1)))


        for i in range(4):
            visualization.make_error_plot(test_error[i], f"M : {self.config[i][0]}  Gamma : {self.config[i][1]}")

        for M, gamma in self.config:
            mlp = MultilayerPerceptron([2, M, 1], 0.01, gamma, 1e5)
            history_error, _ = mlp.train_mlp(10000, self.data, self.labels.view(-1, 1))
            visualization.make_error_plot(history_error, f"M = {M} Gamma = {gamma}")
            visualization.make_decision_boundary_plot(mlp.Wo, mlp.Ws, mlp.Bo, mlp.Bs, min_x, max_x, min_y, max_y, self.class0, self.class1)

# separable_analyze = Analyze(444,DatasetGenerator.generate_linearly_separable)
# separable_analyze.run(1.5, 8.5, 1.5, 8.5)
#
# nonseparable_analyze = Analyze(777,DatasetGenerator.generate_nonlinearly_separable)
# nonseparable_analyze.run(-1.5, 2.5, -1.5, 2.5)

In [10]:
class ImageNormalizer:
    def __call__(self, img):
        X = torch.from_numpy(np.array(img)).float()
        eps = 1e-10
        X_norm = X / (X.max() + eps)
        return X_norm.flatten()

class Dataset:
    def __init__(self, d: int, train_batch: int, test_batch: int, seed: int):
        trans = transforms.Compose([
            transforms.Resize((d,d)),
            ImageNormalizer(),
        ])

        self.dataset = ImageFolder (
            "dataset",
            transform = trans
        )

        train_dataset, test_dataset = random_split(
            self.dataset,
            [0.8, 0.2],
            generator = torch.Generator().manual_seed(seed)
        )

        self.train_loader = DataLoader(
            train_dataset,
            batch_size=train_batch,
            shuffle = False
        )

        self.test_loader = DataLoader(
            test_dataset,
            batch_size=test_batch,
            shuffle=False
        )

In [2]:
def test_normalizer_rango_0_1():
    """La salida ℓ∞ debe quedar en [0, 1]."""
    norm = ImageNormalizer()
    img = Image.fromarray((np.random.rand(16, 16, 3) * 255).astype(np.uint8))
    out = norm(img)
    assert out.min().item() >= 0.0
    assert out.max().item() <= 1.0
    print("OK rango [0,1]:", out.min().item(), out.max().item())

test_normalizer_rango_0_1()

def test_normalizer_max_es_1():
    """Dividir por el máximo => el píxel máximo de la imagen pasa a valer 1."""
    norm = ImageNormalizer()
    arr = np.zeros((8, 8, 3), dtype=np.uint8)
    arr[0, 0] = 200          # un solo píxel con el valor máximo
    out = norm(Image.fromarray(arr))
    assert torch.isclose(out.max(), torch.tensor(1.0), atol=1e-6)
    print("OK max==1:", out.max().item())

test_normalizer_max_es_1()

IndentationError: unindent does not match any outer indentation level (<string>, line 9)

In [ ]:
def loader_to_tensors(loader, k: int):
    """Apila un DataLoader en X [N, D] y T one-hot [N, K]."""
    xs, ys = [], []
    for x_batch, y_batch in loader:
        xs.append(x_batch)
        ys.append(y_batch)

    X = torch.cat(xs).to(dtype=TYPE, device=DEVICE)
    y = torch.cat(ys).to(device=DEVICE).long()

    T = torch.zeros(y.shape[0], k, dtype=TYPE, device=DEVICE)
    T[torch.arange(y.shape[0], device=DEVICE), y] = 1.0
    return X, T

In [ ]:
acrima_configs = [
      {"d": 32, "K": 2},   # D = 32*32*3 = 3072
      {"d": 48, "K": 2},   # D = 48*48*3 = 6912
      {"d": 64, "K": 2},   # D = 64*64*3 = 12288
  ]

P = 50

In [ ]:
def build_acrima_data(d: int, k: int, seed: int = 0, train_batch: int = 128, test_batch: int = 128):
    ds = Dataset(d, train_batch, test_batch, seed)
    X_train, T_train = loader_to_tensors(ds.train_loader, k)
    X_test,  T_test  = loader_to_tensors(ds.test_loader,  k)
    D = X_train.shape[1]
    print(f"d={d}  D={D}  K={k}  | train={X_train.shape[0]}  test={X_test.shape[0]}")
    return (X_train, T_train), (X_test, T_test), D

In [3]:
def make_objective_acrima(X, T, D, M, K, num_epochs: int = P):
    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-6, 1, log=True)
        gamma = trial.suggest_float("gamma", 0.0, 1.0, log=False)

        mlp = MultilayerPerceptron([D, M, K], alpha, gamma, 1000)
        history_error, history_accuracy = mlp.train_mlp(num_epochs, X, T)

        trial.set_user_attr("final_error", history_error[-1])
        trial.set_user_attr("final_accuracy", history_accuracy[-1])
        return history_error[-1]
    return objective

IndentationError: unexpected indent (3034426619.py, line 2)

In [ ]:
def run_acrima(M: int = 64):
    for cfg in acrima_configs:
        (X_tr, T_tr), (X_te, T_te), D = build_acrima_data(cfg["d"], cfg["K"])

        # --- Calibración optuna (γ, α) con P=50 ---
        objective = make_objective_acrima(X_tr, T_tr, D, M, cfg["K"], num_epochs=P)
        trials = study(objective)

        best = min(trials, key=lambda t: t.value)
        alpha, gamma = best.params["alpha"], best.params["gamma"]
        print(f"[d={cfg['d']}] mejor alpha={alpha:.4f} gamma={gamma:.4f} error={best.value:.6f}")

        # --- Entrenamiento final con los mejores hiperparámetros ---
        mlp = MultilayerPerceptron([D, M, cfg["K"]], alpha, gamma, 1000)
        hist_error, hist_acc = mlp.train_mlp(P, X_tr, T_tr)

        # Error en validación (20%)
        mlp.forward(X_te)
        val_error = mlp.get_error(T_te)
        val_acc = mlp.get_accuracy(T_te)
        print(f"[d={cfg['d']}] val_error={val_error:.6f}  val_acc={val_acc:.4f}")

        visualization.make_error_plot(
            hist_error, f"ACRIMA  D={D}  M={M}  α={alpha:.3f}  γ={gamma:.3f}"
        )

  # run_acrima()